In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

: 

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop
# API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError


def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    
    return response.json()

print(get_json(BASE_URL + 'status.json')['service'])
# 'Northern Star Shop API'

print(get_json(BASE_URL + 'orders/page-99.json'))
# HTTPError: 404 Client Error: Not Found for url: https://cvijan.github.io/nss-api/v2/orders/page-99.json


In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа


def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])
        if page['next']:
            url = urljoin(url, page['next'])
        else:
            url = None
    
    return records

orders = fetch_all(BASE_URL + 'orders/page-1.json')
len(orders) # 1852


In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->
# None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

def save_json(records: list, path:str) -> None:
    with open(path, 'w', encoding= 'utf-8') as f:
        json.dump(records, f, ensure_ascii= False)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))

In [0]:
# 4. Preuzimanje fajla, kopiraj bez izmena
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here

    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes,unchanged

    return len(response.content)

download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,
'teams.csv')) 
#801 u data/raw pojavljuje se teams.csv

In [ ]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у
# raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams',
# 'matches'
# 1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
# 2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције download_file
# 3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
# Након тога преименовати колоне
# customer_name у customer
# customer_city у city
# 4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
# По потреби, pd.read_csv(...) или pd.read_parquet(...)
# 5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)


def extract(raw_dir: str) -> dict:
    # Porudzbine
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(RAW_DIR, 'orders.json'))
    
    # Skidanje fajlova
    download_file(FILES_URL + 'products.csv', 
                  os.path.join(RAW_DIR, 'products.csv'))

    download_file(FILES_URL + 'teams.csv', 
                  os.path.join(RAW_DIR, 'teams.csv'))

    download_file(FILES_URL + 'matches.parquet', 
                  os.path.join(RAW_DIR, 'matches.parquet'))

    # Pretvaranje porudzbina u DataFrame
    orders_df = pd.json_normalize(orders, sep='_')

    # Preimenovanje kolona
    orders_df = orders_df.rename(columns={
        'customer_name': 'customer',
        'customer_city': 'city'
    })

    # Citanje fajlova
    products_df = pd.read_csv(os.path.join(RAW_DIR, 'products.csv'))
    teams_df = pd.read_csv(os.path.join(RAW_DIR, 'teams.csv'))
    matches_df = pd.read_parquet(os.path.join(RAW_DIR, 'matches.parquet'))


    return {
        'orders': orders_df,
        'products': products_df,
        'teams': teams_df,
        'matches': matches_df
    }



data = extract(RAW_DIR)

for name,df in data.items():
    print(name, df.shape)

# orders (1852, 8)
# products (108, 7)
# teams (20, 5)
# matches (120, 9)


In [ ]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame
# која чисти каталог производа
# Ако се понавља исти производ, садржати први ред за сваки product_id
# team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред остаје празан (то су производи лиге, без тима)
# category треба да буде без размака, има мала слова
# price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане
# само float (у сваком случају)
# Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
# active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало
# постаје False
# Output: 105 редова, price је типа float64, active је True за 101 производ
# Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves',
# 'price': 1490.0, 'active': True, ...}

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Uklanja ponovljene proizvode
    df = df.drop_duplicates(subset='product_id', keep='first')

    # Ciscenje team_code
    df['team_code'] = (
        df['team_code']
        .fillna('')
        .astype(str)
        .str.strip().upper()
    )
    # Ciscenje category

    df['category'] = (
        df['category']
        .astype(str)
        .str.strip()
        .str.lower()
    )
    # Ciscenje price

    df['price'] = (
        df['price']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(' RSD', '', regex=False)
        .str.replace(' ,', '.', regex=False)
        .to_numeric(df['price'], errors= 'coerce')
    )
    # Ciscenje active

    df['active'] = (
        df['active']
        .astype(str)
        .str.strip()
        .str.lower()
        .eq('yes')
    )


    return df


In [ ]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame
# order_id, customer, city, payment без размака;
# order_id велика слова
# customer и city -> title()
# Град који недостаје (isna) -> 'Unknown'


def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()


    df['order_id'] = (
        df['order_id']
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df['customer'] = (
        df['customer']
        .astype(str)
        .str.strip()
        .title()
    )

    df['city'] = (
        df['city']
        .isna('Unknown')
        .astype(str)
        .str.strip()
        .title()
    )

    df['payment'] = (
        df['payment']
        .astype(str)
        .str.strip()
    )

    return df


In [ ]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    
    unique = df.drop_duplicates(subset='order_id',keep='first').copy()

    duplicates = df[df.duplicated(subset='order_id',keep='first')].copy()

    return unique, duplicates

